# Lab 9.5 &mdash; Challenge: The Service That Is Up and Wrong

**Level:** Advanced &middot; challenge &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Find an incident in which every conventional signal is green
- Show why CPU autoscaling never fires, and pick a signal that does
- Write alarms that catch it, and prove they stay quiet on a good day
- Leave with the runbook page for an agentic service

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, and none of them needs a cluster, so your
> score never depends on a live endpoint or on `kubectl` working. Cells marked **Run it for real**
> do call the sandbox model or your namespace; if either is unreachable they print how to fix it
> instead of crashing.

> **The last lab of the course.** It uses Module 6's citations, Module 7's measurements
> and Module 8's controls, and asks the Module 9 question about all three: how would
> you know, at 09:15, from a dashboard?

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-9-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- graded cells still work)")
print("namespace:", APP_NS or "(unknown -- graded cells still work)")

## Concept

An ordinary service fails by erroring or by slowing down, and both are visible in the four golden
signals &mdash; latency, traffic, errors, saturation. An agent service has a third failure mode:
it answers every request, quickly, with a 200, and the answers are wrong.

Nothing in the golden signals moves. Something else does, and only if you recorded it.

## Section 1 &mdash; Two days that look identical

Here is yesterday and today. Same traffic, same code, one deploy in between.

In [ ]:
import random

def build_day(seed: int, refusal_rate: float, citation_rate: float,
              error_rate: float = 0.02, n: int = 2000) -> list:
    """One day of requests. The same seed gives the same latencies, costs and errors,
    so any difference between two days below is a difference in BEHAVIOUR, not noise."""
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        r = rng.random()
        decision = ("refused" if r < refusal_rate
                    else "escalated" if r < refusal_rate + 0.04
                    else "answered")
        out.append({
            "ok": rng.random() > error_rate,
            "duration_s": round(rng.uniform(1.5, 12.0), 2),
            "cited": rng.random() < citation_rate,
            "cost_usd": round(rng.uniform(0.0008, 0.0032), 5),
            "decision": decision,
        })
    return out


YESTERDAY = build_day(11, refusal_rate=0.08, citation_rate=0.92)
TODAY     = build_day(11, refusal_rate=0.01, citation_rate=0.55)


def pct(values, p):
    """The p-th percentile, nearest-rank. Stdlib, and exact enough for a dashboard."""
    s = sorted(values)
    return s[min(len(s) - 1, max(0, math.ceil(p / 100 * len(s)) - 1))]


def metrics(day: list) -> dict:
    n = len(day)
    return {
        "requests":        n,
        "error_rate":      round(sum(1 for r in day if not r["ok"]) / n, 4),
        "p95_latency_s":   pct([r["duration_s"] for r in day], 95),
        "cost_per_req":    round(sum(r["cost_usd"] for r in day) / n, 5),
        "refusal_rate":    round(sum(1 for r in day if r["decision"] == "refused") / n, 4),
        "escalation_rate": round(sum(1 for r in day if r["decision"] == "escalated") / n, 4),
        "citation_rate":   round(sum(1 for r in day if r["cited"]) / n, 4),
    }


GOLDEN = ("error_rate", "p95_latency_s", "cost_per_req", "requests")
AGENTIC = ("refusal_rate", "escalation_rate", "citation_rate")

print("yesterday:", metrics(YESTERDAY))
print("today    :", metrics(TODAY))

In [ ]:
def moved(before: float, after: float, tolerance: float = 0.25) -> bool:
    """Did this metric change materially between the two windows?"""
    if before == 0:
        return after != 0
    # TODO: a control that STOPS WORKING makes its own metric go down, not up. A test
    # written as `after > before * (1 + tolerance)` finds nothing today. Compare the
    # size of the change, whichever way it went.
    return BLANK


def what_changed(before: dict, after: dict, tolerance: float = 0.25) -> list:
    """Every metric that moved, in the order they are defined."""
    return [k for k in before if moved(before[k], after[k], tolerance)]

In [ ]:
# --- Self-check: Section 1
Y, T = metrics(YESTERDAY), metrics(TODAY)

check("traffic is identical",
      lambda: Y["requests"] == T["requests"])
check("the error rate did not move",
      lambda: not moved(Y["error_rate"], T["error_rate"]))
check("p95 latency did not move",
      lambda: not moved(Y["p95_latency_s"], T["p95_latency_s"]))
check("cost per request did not move",
      lambda: not moved(Y["cost_per_req"], T["cost_per_req"]))
check("NOT ONE OF THE FOUR GOLDEN SIGNALS MOVED",
      lambda: not any(moved(Y[k], T[k]) for k in GOLDEN),
      "every dashboard the team owns is green")
check("the refusal rate collapsed",
      lambda: moved(Y["refusal_rate"], T["refusal_rate"]))
check("...downwards, which is why a one-sided alarm never fired",
      lambda: T["refusal_rate"] < Y["refusal_rate"])
check("the citation rate fell too",
      lambda: moved(Y["citation_rate"], T["citation_rate"]))
check("exactly the agent-specific metrics moved, and only those",
      lambda: set(what_changed(Y, T)) == {"refusal_rate", "citation_rate"})
check("a one-sided test finds nothing at all",
      lambda: [k for k in Y if T[k] > Y[k] * 1.25] == [],
      "which is how this runs for three weeks")

def _diff():
    print(f"  {'metric':18} {'yesterday':>10} {'today':>10}   moved?")
    for k in Y:
        flag = "  <-- MOVED" if moved(Y[k], T[k]) else ""
        print(f"  {k:18} {Y[k]:>10} {T[k]:>10}{flag}")
guard(_diff)

### What happened

A deploy changed a prompt. The guardrail that used to hold sanctions cases for a human now
answers most of them, and the retriever's grounding check stopped rejecting ungrounded answers.

The service is up. It is fast. It costs the same. It answers every request with a 200, and one
payment in twelve that should have gone to a human did not.

This is the failure mode Module 8 closed on, seen from the dashboard, and the reason those two
metrics have to exist as **first-class signals with alarms on them**, next to latency and errors
rather than in a weekly report.

## Section 2 &mdash; The signal that actually moves with load

The second half of the incident: at 09:15 the same service went from four concurrent requests to
sixty, and the HPA did nothing at all.

In [ ]:
CALL_SECONDS   = 8.0      # one agent request, mostly spent waiting on the gateway.
                          # Measured on this sandbox: 7.5-10s for a one-line answer.
CPU_PER_REQ    = 0.015    # the CPU it actually uses
PER_REPLICA    = 8        # concurrent requests one replica serves without queueing
TARGET_UTIL    = 0.70

def cpu_percent(in_flight: int) -> float:
    """CPU utilisation of the fleet's replicas at this concurrency."""
    return 100.0 * in_flight * CPU_PER_REQ / CALL_SECONDS


def replicas_from_cpu(in_flight: int, current: int = 1, target: int = 70) -> int:
    """What an HPA on CPU utilisation asks for. This is the shipped default."""
    util = cpu_percent(in_flight) / current
    return max(1, math.ceil(current * util / target))


def replicas_from_inflight(in_flight: int) -> int:
    """What an HPA on in-flight requests asks for."""
    # TODO: each replica serves PER_REPLICA concurrent requests, and you do not want them
    # running at 100% -- aim for TARGET_UTIL of that. Return the replica count, never
    # fewer than one.
    return BLANK


def latency_at(in_flight: int, replicas: int) -> float:
    """Wall clock per request once the queue forms. Crude, and the right shape."""
    capacity = replicas * PER_REPLICA
    return CALL_SECONDS * math.ceil(max(1, in_flight) / capacity)

In [ ]:
# --- Self-check: Section 2
check("at four in flight one replica is right, and both rules agree",
      lambda: replicas_from_cpu(4) == 1 and replicas_from_inflight(4) == 1)
check("at sixty in flight the CPU rule still asks for one",
      lambda: replicas_from_cpu(60) == 1)
check("...because the fleet is under 16% busy while it queues",
      lambda: cpu_percent(60) < 16)
check("THE IN-FLIGHT RULE ASKS FOR ELEVEN",
      lambda: replicas_from_inflight(60) == 11)
check("one replica at sixty in flight is a 64-second request",
      lambda: latency_at(60, 1) == 64.0)
check("eleven replicas bring it back to one call time",
      lambda: latency_at(60, replicas_from_inflight(60)) == CALL_SECONDS)
check("the CPU rule leaves latency 8x worse than the in-flight rule",
      lambda: latency_at(60, replicas_from_cpu(60))
              == 8 * latency_at(60, replicas_from_inflight(60)))
check("neither rule scales down below one replica",
      lambda: replicas_from_cpu(0) == 1 and replicas_from_inflight(0) == 1)
check("the in-flight rule is bounded by maxReplicas, which is a budget decision",
      lambda: min(replicas_from_inflight(400), 3) == 3,
      "at 400 in flight it wants 72; your quota says 3, so the answer is a queue and a 429")

def _scaling():
    print(f"  {'in flight':>10} {'CPU %':>7} {'cpu rule':>9} {'inflight rule':>14} "
          f"{'latency (cpu)':>14} {'latency (inflight)':>19}")
    for n in (4, 12, 30, 60, 120):
        rc, ri = replicas_from_cpu(n), replicas_from_inflight(n)
        print(f"  {n:>10} {cpu_percent(n):>6.1f}% {rc:>9} {ri:>14} "
              f"{latency_at(n, rc):>13.0f}s {latency_at(n, ri):>18.0f}s")
guard(_scaling)

### Read it

The HPA in the starter manifest &mdash; and in most agent deployments &mdash; scales on CPU at
70%. On this workload it reaches 16% at sixty concurrent requests, so it never fires, and the
replica sitting at 16% busy is serving 64-second requests.

The fix is not a lower CPU target. It is a **different signal**: in-flight requests, queue depth,
or time-to-first-token, exported by your own service and scraped as a custom metric. All three
grow with load because all three are about waiting, which is what this service does.

And note the last check. Scaling has a ceiling that is a budget, not a technical limit. Past it,
the correct behaviour is to shed load with a `429` and a `Retry-After`, not to accept a request
you will answer in four minutes.

## Section 3 &mdash; Alarms that would have caught it

An alarm has two jobs, and the second one is why most alarms get switched off: fire on the
incident, and stay quiet on every good day.

In [ ]:
def alarm_golden_signals(before: dict, after: dict) -> bool:
    """The alarms the team already has. Provided so you can see them not fire."""
    return any(moved(before[k], after[k]) for k in GOLDEN)


def alarm_control_drift(before: dict, after: dict) -> bool:
    """A control's own metric moved. Either direction, because down is the dangerous one."""
    return any(moved(before[k], after[k]) for k in AGENTIC)


def alarm_saturation(in_flight: int, replicas: int) -> bool:
    """Fires while requests are queueing, whatever the CPU says."""
    # TODO: the fleet can serve replicas x PER_REPLICA concurrent requests. Fire when
    # in-flight is above the share of that you are willing to run at (TARGET_UTIL).
    return BLANK

In [ ]:
# --- Self-check: Section 3
QUIET_DAY = metrics(build_day(12, refusal_rate=0.08, citation_rate=0.92))

check("the alarms the team already has do not fire on the incident",
      lambda: alarm_golden_signals(Y, T) is False,
      "this is not a criticism of them -- they are measuring something else")
check("THE CONTROL-DRIFT ALARM FIRES",
      lambda: alarm_control_drift(Y, T) is True)
check("...and stays quiet comparing two ordinary days",
      lambda: alarm_control_drift(Y, QUIET_DAY) is False,
      "an alarm that fires on a good day is an alarm somebody mutes")
check("the golden-signal alarms are also quiet on a good day",
      lambda: alarm_golden_signals(Y, QUIET_DAY) is False)
check("the saturation alarm fires at sixty in flight on one replica",
      lambda: alarm_saturation(60, 1) is True)
check("...and not once it has scaled out",
      lambda: alarm_saturation(60, replicas_from_inflight(60)) is False)
check("it fires before latency doubles, not after",
      lambda: alarm_saturation(9, 1) is True and latency_at(9, 1) == 2 * CALL_SECONDS)
check("a CPU alarm at 70% is silent at every concurrency worth alarming on",
      lambda: all(cpu_percent(n) < 70 for n in (10, 60, 120, 200)),
      "it crosses 70% only near 400 in flight, where a request already takes 400 seconds")

def _alarms():
    for label, pair in (("incident (yesterday -> today)", (Y, T)),
                        ("ordinary day vs ordinary day",  (Y, QUIET_DAY))):
        print(f"  {label:32} golden={str(alarm_golden_signals(*pair)):5} "
              f"control_drift={alarm_control_drift(*pair)}")
    print()
    for n, reps in ((4, 1), (60, 1), (60, 11)):
        print(f"  {n:>3} in flight on {reps:>2} replica(s): saturation="
              f"{str(alarm_saturation(n, reps)):5} latency={latency_at(n, reps):.0f}s")
guard(_alarms)

## The runbook page

Everything above is one page of an on-call runbook. Yours will differ; the shape will not.

**Page on**

| Signal | Threshold | Because |
|---|---|---|
| 5xx rate | above 1% for 5 min | the ordinary one; keep it |
| p95 latency | above 3&times; the baseline | the ordinary one; keep it |
| in-flight per replica | above 70% of capacity | CPU will not tell you this |
| refusal / escalation rate | moved &plusmn;25% vs the last 7 days | **a control stopped working** |
| citation rate | moved &plusmn;25% | grounding stopped working |
| cost per request | above 2&times; the baseline | a retry loop, or a routing change |

**First three things to do**

1. **Read one trace, not the logs.** Find a slow or wrong request by trace ID and look at the
   span tree. Which hop grew, and did it grow in count or in duration?
2. **Compare the last deploy.** A prompt is a deploy. So is a model version change made by
   somebody else on the gateway you depend on.
3. **Check the dependency before restarting anything.** Lab 9.2's whole point: restarting a
   healthy process because a remote gateway blinked makes the outage longer.

**Do not**

- Do not raise the CPU target to make the HPA fire. It is the wrong signal, not a mistuned one.
- Do not turn off the control that is alarming. Its metric moving is the alarm.
- Do not conclude anything from a green dashboard. Today's incident had one.

## Run it for real

Your own numbers, from the sandbox gateway. Six sequential calls, then the replica count the
in-flight rule would ask for at sixty concurrent users on *your* measured latency.

In [ ]:
if llm_ready():
    def _budget():
        lat = []
        for i in range(6):
            t0 = time.perf_counter()
            ask(f"In one sentence, what is a payment exception? (v{i})")
            lat.append(time.perf_counter() - t0)
        p95 = pct(lat, 95)
        per_replica = max(1, int(PER_REPLICA))
        want = math.ceil(60 / (per_replica * TARGET_UTIL))
        print(f"  measured  : mean {sum(lat) / len(lat):.1f}s, p95 {p95:.1f}s over 6 calls")
        print(f"  at 60 concurrent users the in-flight rule asks for {want} replicas")
        print(f"  the CPU rule asks for {replicas_from_cpu(60)}")
        print(f"  and one replica would answer in about {latency_at(60, 1):.0f}s")
        print("\n  Six calls is not a latency distribution. It is enough to know which")
        print("  order of magnitude you are budgeting in, which is the decision here.")
    guard(_budget)

In [ ]:
score()

## Your turn

1. The control-drift alarm compares two windows. Write the version that compares today with a
   **trailing seven-day median**, and work out what it does on the Monday after a long weekend.
2. Add a `429` path to Lab 9.1's `handle`: shed load when in-flight is above capacity, with a
   `Retry-After`. Then decide which is worse for your callers &mdash; a 429 now, or a 200 in
   four minutes.
3. Take one control from Module 8 &mdash; the approval gate, the contract, the detector &mdash;
   and write the metric that proves it is still running. If you cannot, that control is
   unmonitored, and Section 1 is what that looks like on the day it stops.

**What you take from Module 9:** a service boundary that returns its failures, probes that answer
two different questions, a readiness checklist that executes, spans that carry cost and decisions,
and the two signals &mdash; control drift and saturation &mdash; that an agent needs and a web
service does not.

That is the last lab. The capstone puts all nine modules behind one endpoint.